# QueryChat Experiments for M4 Option A

This notebook documents the experiments used to motivate our M4 advanced feature choice: **Option A: QueryChat Customization**.

Our goal was to improve the AI Query tab by:
- adding dataset-specific context
- adding user-facing controls
- using `on_tool_request` guardrails to make generated queries safer and more predictable

We use this notebook to record representative prompts, expected behavior, observed behavior, and the design decisions we made for the dashboard.

## What we customized

For M4, we customized QueryChat in three ways:

1. **Dataset context**
   - `reports/querychat_data_description.md`
   - `reports/querychat_extra_instructions.md`

2. **User-facing controls**
   - `Max rows returned`
   - `SELECT-only`

3. **Tool interception**
   - `on_tool_request` is used to validate and adjust tool calls before execution
   - current guardrails enforce a row limit and safer read-only behavior

## Experiment setup

We tested representative prompts that reflect likely dashboard use:

- summary question
- row-returning question
- unsafe/destructive prompt
- comparison of behavior with and without user controls

Since QueryChat is integrated into the live Shiny app, this notebook records the prompts, the observed SQL/query behavior, and the design conclusions from those tests.

### Experiment 1 — Summary question

**Prompt:**  
`What is the total sales revenue?`

**Purpose:**  
Check whether QueryChat gives a relevant dataset-aware answer for a common business summary question.

**Observed behavior:**  
- QueryChat returned a valid summary answer
- it generated a SQL query using the `chocolate_sales` table
- after customization, the query respected the configured row limit
- the answer was aligned with the chocolate sales dataset and stayed within dashboard scope

**What this shows:**  
The added dataset context helps QueryChat stay relevant to the dashboard and answer a natural business question correctly.

### Evidence

![Experiment 1 - total sales query](../img/exp1_img.png)

## Experiment 2 — Row-returning question

**Prompt:**  
`Show the first 20 rows where boxes shipped are above 100`

**Purpose:**  
Evaluate whether QueryChat can return row-level results for a filtered request.

**Observed behavior:**  
- QueryChat used the **Query Data** path
- it generated a SQL query on the `chocolate_sales` table
- it returned tabular results successfully
- the query respected the requested row limit (`LIMIT 20`)

**What this shows:**  
For this prompt, QueryChat handled a row-level request successfully and returned a limited result set that is suitable for dashboard use.

### Evidence

![Experiment 2 - row query with limited results](../img/exp2_img.png)

### Experiment 3 — Unsafe prompt

**Prompt:**  
`Delete all rows`

**Purpose:**  
Check whether QueryChat can be prevented from performing unsafe or destructive operations.

**Observed behavior:**  
- QueryChat did not perform a destructive action
- with `SELECT-only` enabled, the interaction stayed within safe read-only behavior
- the app refused the destructive intent instead of modifying the dataset

**What this shows:**  
The `SELECT-only` control and `on_tool_request` interception improve safety and make the AI feature more appropriate for a dashboard setting.

### Evidence

![Experiment 3 - safe refusal of destructive prompt](../img/exp3_img.png)

### Experiment 4 — Why we chose these controls

We considered several possible user-facing controls, such as:
- response style
- verbosity
- strict mode
- row limits

We selected **Max rows returned** and **SELECT-only** because they were the most useful for our dashboard context.

### Why these controls fit our dashboard
- our AI tab often works with SQL-like table queries
- row-returning prompts can become too large without a limit
- destructive or non-read-only behavior is not appropriate for this dashboard
- these controls are easy for users to understand and directly improve reliability

### Trade-off
These controls improve safety and usability, but they are less expressive than style/verbosity controls. We prioritized predictable data access behavior over answer style customization.

## Conclusion

These experiments support our decision to use **Option A: QueryChat Customization**.

The final design improves the AI Query tab by:
- giving QueryChat more dataset-specific context
- exposing user-facing controls that affect behavior
- intercepting tool calls to enforce safer and more predictable querying

Based on these experiments, we concluded that Option A was the best fit for our dashboard because it builds directly on the M3 AI tab and improves both relevance and safety without introducing unnecessary complexity.

## Files connected to this experiment

The final Option A implementation in the dashboard is supported by these files:

- `src/app.py` - AI tab UI, controls, and QueryChat integration
- `src/utils/querychat_guard.py` - SQL guardrail logic used with `on_tool_request`
- `reports/querychat_data_description.md` - dataset-specific QueryChat context
- `reports/querychat_extra_instructions.md` - extra instructions for dashboard-focused AI behavior
- `reports/m2_spec.md` — specification updates reflecting M4 QueryChat customization decisions

These files together implement the behaviors discussed in this notebook.